# WELLGen — WELL512a equidistribution

**WELL** (Well Equidistributed Long-period Linear, Panneton-L'Ecuyer-
Matsumoto 2006) addresses MT's slow recovery from zero-heavy states
and its weaker low-resolution equidistribution. The canonical WELL512a
parameter set has $k = wr - p = 32 \cdot 16 = 512$ and is reported
maximally equidistributed (ME) at $L = 32$.

The family is parameterised by eight matrix recipes $T_0, \ldots, T_7$
(each picked from a small algebra $\{M_0, M_1, M_2, M_3, M_4, M_5\}$)
and three offsets $m_1, m_2, m_3$. See
[generators/WELLGen.md](../../generators/WELLGen.md).


## Imports


In [ ]:
# stamp:auto-generated
from regpoly import make_combined
from regpoly.core.generator import Generator
from regpoly.core.transformation import Transformation
from regpoly.analyses.equidistribution_test import (
    EquidistributionTest,
    METHOD_MATRICIAL, METHOD_HARASE,
    METHOD_NOTPRIMITIVE, METHOD_SIMD_NOTPRIMITIVE,
)

INT_MAX = 2**31 - 1


## Construct the generator — _Panneton, L'Ecuyer & Matsumoto (2006)_


In [ ]:
# Canonical WELL512a parameters (Panneton-L'Ecuyer-Matsumoto 2006).
gen = Generator.create("WELLGen", L=32,
    w=32, r=16, p=0, m1=13, m2=9, m3=5,
    matrices={
        "T0": {"M": 3, "t": -16},
        "T1": {"M": 3, "t": -15},
        "T2": {"M": 3, "t":  11},
        "T3": {"M": 0},
        "T4": {"M": 3, "t":  -2},
        "T5": {"M": 3, "t": -18},
        "T6": {"M": 2, "t": -28},
        "T7": {"M": 5, "b": 0xDA442D24, "t": -5},
    })
print(gen.display())


## Wrap in a `CombinedF2LinearSource`


In [ ]:
# Wrap the generator in a single-component CombinedF2LinearSource so
# the equidistribution test consumes the same shape every search-loop
# candidate has.
comb = make_combined(gen, Lmax=gen.L)
print(f"k_g = {comb.k()}, L = {comb.L()}")


## Equidistribution test

WELL is full-period, so the **Harase** method applies.


In [ ]:
# Build the equidistribution test and run it. We cap `delta` at
# INT_MAX so nothing is rejected — we just want the score.
test = EquidistributionTest(
    L=gen.L,
    delta=[INT_MAX] * (gen.L + 1),
    mse=INT_MAX,
    method=METHOD_HARASE,
)
result = test.run(comb)

print(f"SE (Σ gaps)   = {result.se}")
print(f"verified      = {result.verified}")
print(f"first 10 gaps = {[result.ecart[i] for i in range(1, min(11, gen.L + 1))]}")


## Catalog entry

The published version of this parameter set lives in the REGPOLY catalog under `library_id = "well512a"`. To load it programmatically without hard-coding parameters:

```python
from regpoly.library import Catalog
cat = Catalog('docs/library')
cat.load()
_, entry = cat.generator('well512a')
# entry.components[0] carries the same params as constructed above
```
